# Unit 9: BERT workflows for text analysis

There are **three core workflows** for text in computational social science (slide 7):

1. **Sentiment analysis** — what's the *tone* of this text?
2. **Topic modeling** — what's being discussed across many texts?
3. **Labeling** — given a label set, which one fits each text?

Today's notebook covers **(1) sentiment** and **(2) topic modeling**, comparing a classical method to a BERT-based one in each case:

| Workflow | Classical | BERT-based |
|----------|-----------|------------|
| Sentiment | VADER (dictionary) | `pipeline('sentiment-analysis')` |
| Topic modeling | LDA (gensim) | BERTopic |

**Workflow (3) — zero-shot labeling — is in a separate notebook (slide 39).**

In [1]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import pipeline

import pandas as pd
from sklearn.datasets import fetch_20newsgroups
import gensim
from gensim import corpora
from gensim.models.ldamodel import LdaModel
from bertopic import BERTopic

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline

from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

import pyLDAvis.gensim_models as gensimvis
import pyLDAvis

**One-time downloads** for NLTK (tokenizer + stopword list).

In [2]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/mingyuezha/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/mingyuezha/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/mingyuezha/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [11]:
df = pd.read_csv("data/df.csv")
df = df[df["has_ai_overview"] == True]
df = df[df["location"] == "United States"]

## Comparing Sentiment

### Vader

In [12]:
# Initialize VADER Sentiment Analyzer
vader_analyzer = SentimentIntensityAnalyzer()
df["vaderSent"] = df.all_items.apply(lambda x: vader_analyzer.polarity_scores(x))

In [13]:
df

,Unnamed: 0.1,Unnamed: 0,location,query,has_ai_overview,all_items,cited_sources,all_references,paa_questions,platform,topic,section,query_en,query_zh,related_searches,vaderSent
0,0,0.0,United States,breast cancer,True,['A new lump or mass in the breast or underarm...,[{'title': 'Breast cancer - World Health Organ...,[{'title': 'Breast cancer - Symptoms and cause...,"['What are the top 3 signs of breast cancer?',...",google,cancer,common_conditions,NaN,NaN,NaN,"{'neg': 0.189, 'neu': 0.731, 'pos': 0.08, 'com..."
1,1,1.0,United States,what is cancer,True,['Tumor Formation: Extra cells form masses of ...,"[{'title': 'What Is Cancer? Symptoms, Causes &...",[{'title': 'What Is Cancer? - NCI - National C...,"['What is the simple definition of cancer?', '...",google,cancer,common_conditions,NaN,NaN,NaN,"{'neg': 0.119, 'neu': 0.788, 'pos': 0.094, 'co..."
2,2,2.0,United States,cancer symptoms,True,['Unexplained Weight Loss: Losing $\\ge 10$ po...,[{'title': 'Cancer - Symptoms and causes - May...,[{'title': 'Cancer - Symptoms and causes - May...,"['What are the 7 warning signs for cancer?', '...",google,cancer,common_conditions,NaN,NaN,NaN,"{'neg': 0.193, 'neu': 0.807, 'pos': 0.0, 'comp..."
3,3,3.0,United States,lung cancer,True,['Non-Small Cell Lung Cancer (NSCLC): Accounts...,[{'title': 'Lung cancer - World Health Organiz...,[{'title': 'What Is Lung Cancer? | Types of Lu...,['How long can you live with one lung cancer?'...,google,cancer,common_conditions,NaN,NaN,NaN,"{'neg': 0.242, 'neu': 0.702, 'pos': 0.056, 'co..."
4,4,4.0,United States,skin cancer,True,"[""Basal Cell Carcinoma (BCC): The most common ...","[{'title': 'Skin Cancer Basics - CDC', 'link':...",[{'title': 'Skin cancer - Symptoms and causes ...,['What does skin cancer look like when it just...,google,cancer,common_conditions,NaN,NaN,NaN,"{'neg': 0.068, 'neu': 0.907, 'pos': 0.025, 'co..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,112,112.0,United States,herbal chinese medicine,True,['Synergistic Formulas: Herbs are rarely presc...,[{'title': 'Chinese herbal medicine | Better H...,[{'title': 'Traditional Chinese Medicine: What...,"['What are Chinese herbal medicines?', 'Which ...",google,herbal,tcm,NaN,NaN,NaN,"{'neg': 0.103, 'neu': 0.767, 'pos': 0.13, 'com..."
113,113,113.0,United States,chinese herbal medicine,True,[],[],[],[],google,herbal,tcm,NaN,NaN,NaN,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound..."
114,114,114.0,United States,supreme court abortion,True,"['The Dobbs Decision: In 2022, the Court ruled...",[{'title': 'Roe v. Wade and Supreme Court Abor...,[{'title': 'Roe v. Wade and Supreme Court Abor...,['What is the new Supreme Court decision on ab...,google,abortion,legally_variable,NaN,NaN,NaN,"{'neg': 0.069, 'neu': 0.833, 'pos': 0.098, 'co..."
115,115,115.0,United States,hiv treatment,True,['Undetectable = Untransmittable (U=U): When y...,"[{'title': 'Treating HIV - CDC', 'link': 'http...","[{'title': 'Treating HIV - CDC', 'link': 'http...","['Can HIV be treated completely?', 'What are 7...",google,hiv,legally_variable,NaN,NaN,NaN,"{'neg': 0.033, 'neu': 0.903, 'pos': 0.064, 'co..."


### BERT

**Why 500 characters?** BERT models have a **512-token input limit** — anything longer gets dropped. Truncating to 500 characters is a quick way to stay under that for our roughly-word-per-character English text.

`pipeline('sentiment-analysis')` defaults to **`distilbert-base-uncased-finetuned-sst-2-english`** — a smaller distilled BERT fine-tuned on the Stanford Sentiment Treebank. Output is `{'label': 'POSITIVE'/'NEGATIVE', 'score': float}`.

In [14]:
bert_classifier = pipeline('sentiment-analysis')
bert_scores = bert_classifier(df.all_items.tolist())

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


In [15]:
df["bertSent"] =  bert_scores

In [16]:
df.head()

,Unnamed: 0.1,Unnamed: 0,location,query,has_ai_overview,all_items,cited_sources,all_references,paa_questions,platform,topic,section,query_en,query_zh,related_searches,vaderSent,bertSent
0,0,0.0,United States,breast cancer,True,['A new lump or mass in the breast or underarm...,[{'title': 'Breast cancer - World Health Organ...,[{'title': 'Breast cancer - Symptoms and cause...,"['What are the top 3 signs of breast cancer?',...",google,cancer,common_conditions,NaN,NaN,NaN,"{'neg': 0.189, 'neu': 0.731, 'pos': 0.08, 'com...","{'label': 'NEGATIVE', 'score': 0.9954103827476..."
1,1,1.0,United States,what is cancer,True,['Tumor Formation: Extra cells form masses of ...,"[{'title': 'What Is Cancer? Symptoms, Causes &...",[{'title': 'What Is Cancer? - NCI - National C...,"['What is the simple definition of cancer?', '...",google,cancer,common_conditions,NaN,NaN,NaN,"{'neg': 0.119, 'neu': 0.788, 'pos': 0.094, 'co...","{'label': 'NEGATIVE', 'score': 0.9930496215820..."
2,2,2.0,United States,cancer symptoms,True,['Unexplained Weight Loss: Losing $\\ge 10$ po...,[{'title': 'Cancer - Symptoms and causes - May...,[{'title': 'Cancer - Symptoms and causes - May...,"['What are the 7 warning signs for cancer?', '...",google,cancer,common_conditions,NaN,NaN,NaN,"{'neg': 0.193, 'neu': 0.807, 'pos': 0.0, 'comp...","{'label': 'NEGATIVE', 'score': 0.999496579170227}"
3,3,3.0,United States,lung cancer,True,['Non-Small Cell Lung Cancer (NSCLC): Accounts...,[{'title': 'Lung cancer - World Health Organiz...,[{'title': 'What Is Lung Cancer? | Types of Lu...,['How long can you live with one lung cancer?'...,google,cancer,common_conditions,NaN,NaN,NaN,"{'neg': 0.242, 'neu': 0.702, 'pos': 0.056, 'co...","{'label': 'NEGATIVE', 'score': 0.9962447285652..."
4,4,4.0,United States,skin cancer,True,"[""Basal Cell Carcinoma (BCC): The most common ...","[{'title': 'Skin Cancer Basics - CDC', 'link':...",[{'title': 'Skin cancer - Symptoms and causes ...,['What does skin cancer look like when it just...,google,cancer,common_conditions,NaN,NaN,NaN,"{'neg': 0.068, 'neu': 0.907, 'pos': 0.025, 'co...","{'label': 'NEGATIVE', 'score': 0.9992039799690..."


In [17]:
df['bertSent'].apply(lambda x: x['score'])

0      0.995410
1      0.993050
2      0.999497
3      0.996245
4      0.999204
         ...   
112    0.916854
113    0.913810
114    0.997806
115    0.995533
117    0.812691
Name: bertSent, Length: 89, dtype: float64

### Check the correlation between the two of these sentiment analyses pipelines

**Hint:** the two outputs aren't directly comparable. Pull a single signed score from each:

- VADER: `vaderSent['compound']` is already in [−1, +1].
- BERT: combine `label` and `score` — `score` if positive, `−score` if negative.

Then `.corr()` on the two columns.

In [18]:
df['vader_score'] = df['vaderSent'].apply(lambda x: x['compound'])
df['bert_score'] = df['bertSent'].apply(lambda x: -x['score'] if x['label'] == 'NEGATIVE' else x['score'])
df.head()

df[['vader_score', 'bert_score']].corr()

,vader_score,bert_score
vader_score,1.000000,0.199713
bert_score,0.199713,1.000000


Find an examples where the two sentiment classifications are opposing. Why might this be the case?

**Hint:** filter for rows where one score is strongly positive and the other is strongly negative — e.g., `(vader_score > 0.5) & (bert_score < -0.5)` (or the reverse).

In [19]:
a = df[(df['vader_score'] > 0.5) & (df['bert_score'] < -0.5)]
b = df[(df['vader_score'] < 0.5) & (df['bert_score'] > -0.5)]
pd.concat([a, b], ignore_index=True)

,Unnamed: 0.1,Unnamed: 0,location,query,has_ai_overview,all_items,cited_sources,all_references,paa_questions,platform,topic,section,query_en,query_zh,related_searches,vaderSent,bertSent,vader_score,bert_score
0,13,13.0,United States,diabetes type 1,True,['Cause: The immune system mistakenly attacks ...,"[{'title': 'Type 1 Diabetes: What It Is, Cause...",[{'title': 'Type 1 diabetes - Symptoms and cau...,['Can a person with type 1 diabetes live a nor...,google,diabetes,common_conditions,NaN,NaN,NaN,"{'neg': 0.114, 'neu': 0.731, 'pos': 0.155, 'co...","{'label': 'NEGATIVE', 'score': 0.9895226359367...",0.8141,-0.989523
1,24,24.0,United States,hypertension symptoms,True,"['Severe headaches', 'Shortness of breath', 'V...",[{'title': '9 Surprisingly Subtle Symptoms of ...,[{'title': 'High blood pressure (hypertension)...,"['What are 5 signs of hypertension?', 'What ar...",google,hypertension,common_conditions,NaN,NaN,NaN,"{'neg': 0.122, 'neu': 0.706, 'pos': 0.171, 'co...","{'label': 'NEGATIVE', 'score': 0.9982329607009...",0.7096,-0.998233
2,30,30.0,United States,how does gender reassignment surgery work,True,['Transfeminine (Male-to-Female): Procedures l...,[{'title': 'Gender Affirming Surgery Program |...,[{'title': 'Trans-feminine (Male to Female) Su...,['How does gender reassignment surgery male to...,google,gender_affirming,legally_variable,NaN,NaN,NaN,"{'neg': 0.0, 'neu': 0.959, 'pos': 0.041, 'comp...","{'label': 'NEGATIVE', 'score': 0.9901949763298...",0.5719,-0.990195
3,31,31.0,United States,hiv testing,True,['Antibody Tests: Look for antibodies your bod...,"[{'title': 'Getting Tested for HIV - CDC', 'li...","[{'title': 'Getting Tested for HIV - CDC', 'li...","['How are you tested for HIV?', 'Can you have ...",google,hiv,legally_variable,NaN,NaN,NaN,"{'neg': 0.0, 'neu': 0.917, 'pos': 0.083, 'comp...","{'label': 'NEGATIVE', 'score': 0.9071069955825...",0.9186,-0.907107
4,33,33.0,United States,what hiv,True,['Unprotected Sex: Having anal or vaginal sex ...,"[{'title': 'What Are HIV and AIDS?', 'link': '...","[{'title': 'About HIV - CDC', 'link': 'https:/...","['What is HIV caused by?', 'What happens when ...",google,hiv,legally_variable,NaN,NaN,NaN,"{'neg': 0.058, 'neu': 0.798, 'pos': 0.143, 'co...","{'label': 'NEGATIVE', 'score': 0.9923146367073...",0.8360,-0.992315
5,35,35.0,United States,depression help,True,['Call or text 988 to reach the 988 Suicide & ...,[{'title': 'Mental Health Conditions: Depressi...,[{'title': 'Major Depression | Johns Hopkins M...,"['How do I get myself out of depression?', 'Ho...",google,depression,controversial,NaN,NaN,NaN,"{'neg': 0.104, 'neu': 0.722, 'pos': 0.174, 'co...","{'label': 'NEGATIVE', 'score': 0.9260169863700...",0.8168,-0.926017
6,39,39.0,United States,massage cupping therapy,True,"[""Improved Circulation: The suction draws bloo...","[{'title': 'Cupping Therapy: Definition, Types...","[{'title': 'Cupping Therapy: Definition, Types...","['What does massage cupping do?', ""Why can't y...",google,cupping,tcm,NaN,NaN,NaN,"{'neg': 0.076, 'neu': 0.825, 'pos': 0.099, 'co...","{'label': 'NEGATIVE', 'score': 0.5176575779914...",0.7269,-0.517658
7,62,62.0,United States,long covid treatment,True,['Medications: Over-the-counter pain relievers...,[{'title': 'Post Covid Treatment Options - Riv...,"[{'title': 'Long COVID - Cleveland Clinic', 'l...","[""What's the best treatment for long covid?"", ...",google,long_covid,controversial,NaN,NaN,NaN,"{'neg': 0.124, 'neu': 0.69, 'pos': 0.187, 'com...","{'label': 'NEGATIVE', 'score': 0.9955297112464...",0.8343,-0.995530
8,69,69.0,United States,hiv test,True,['Nucleic Acid Test (NAT): Detects the virus d...,"[{'title': 'HIV Testing | HIV Partners - CDC',...","[{'title': 'Getting Tested for HIV - CDC', 'li...","['How do they test for HIV?', 'What are the 7 ...",google,hiv,legally_variable,NaN,NaN,NaN,"{'neg': 0.0, 'neu': 0.921, 'pos': 0.079, 'comp...","{'label': 'NEGATIVE', 'score': 0.9972147345542...",0.8834,-0.997215
9,70,70.0,United States,what is herbal m

# Topic Modeling

### Latent Dirichlet Allocation

In [20]:
num_topics = 10

In [21]:
list_stopwords = stopwords.words("english")
porter = PorterStemmer()

def process_step(one_str):   
    nostop_listing = [word for word in wordpunct_tokenize(one_str)
                      if word not in list_stopwords]
    clean_listing = [porter.stem(word) for word in nostop_listing
                    if word.isalpha() 
                    and len(word) > 3]
    clean_listing_str = " ".join(clean_listing)
    return(clean_listing_str)
    
df["text_proc"] = df.all_items.apply(process_step)

In [22]:
# Preprocess data
documents_clean = [gensim.utils.simple_preprocess(doc) for doc in df.text_proc]
dictionary = corpora.Dictionary(documents_clean)
corpus = [dictionary.doc2bow(doc) for doc in documents_clean]

In [23]:
lda_model = LdaModel(corpus=corpus, num_topics=10, id2word=dictionary, random_state=42)

In [24]:
topics = lda_model.print_topics(num_words = 10)

for topic in topics:
    print(topic)


(0, '0.010*"symptom" + 0.009*"fatigu" + 0.008*"mental" + 0.008*"health" + 0.007*"blood" + 0.007*"bodi" + 0.007*"chang" + 0.007*"loss" + 0.007*"persist" + 0.007*"breath"')
(1, '0.012*"hypertens" + 0.010*"medic" + 0.010*"heart" + 0.010*"mmhg" + 0.010*"chang" + 0.008*"blood" + 0.007*"loss" + 0.007*"immedi" + 0.006*"reduc" + 0.006*"physic"')
(2, '0.009*"symptom" + 0.008*"week" + 0.007*"case" + 0.006*"ill" + 0.005*"day" + 0.005*"health" + 0.005*"use" + 0.005*"requir" + 0.005*"last" + 0.004*"within"')
(3, '0.019*"medic" + 0.009*"use" + 0.008*"undetect" + 0.007*"bodi" + 0.007*"viru" + 0.007*"therapi" + 0.007*"prevent" + 0.006*"includ" + 0.006*"share" + 0.005*"surgeri"')
(4, '0.025*"test" + 0.010*"antibodi" + 0.009*"like" + 0.008*"locat" + 0.008*"find" + 0.008*"day" + 0.008*"health" + 0.007*"blood" + 0.007*"exposur" + 0.006*"look"')
(5, '0.015*"blood" + 0.010*"cancer" + 0.010*"cell" + 0.009*"insulin" + 0.009*"urin" + 0.008*"like" + 0.007*"frequent" + 0.006*"diet" + 0.006*"extrem" + 0.006*"suga

In [25]:
pyLDAvis.enable_notebook()

### visualize
lda_display = gensimvis.prepare(lda_model, corpus, dictionary)
pyLDAvis.display(lda_display)

In [26]:
corpus[0]

[(0, 1),
 (1, 3),
 (2, 2),
 (3, 1),
 (4, 1),
 (5, 1),
 (6, 1),
 (7, 1),
 (8, 6),
 (9, 4),
 (10, 3),
 (11, 1),
 (12, 1),
 (13, 1),
 (14, 1),
 (15, 1),
 (16, 1),
 (17, 1),
 (18, 1),
 (19, 1),
 (20, 2),
 (21, 1),
 (22, 1),
 (23, 1),
 (24, 1),
 (25, 1),
 (26, 1),
 (27, 1),
 (28, 1),
 (29, 2),
 (30, 1),
 (31, 1),
 (32, 1),
 (33, 1),
 (34, 2),
 (35, 1),
 (36, 1),
 (37, 3),
 (38, 1),
 (39, 1),
 (40, 1),
 (41, 1),
 (42, 1),
 (43, 2),
 (44, 1),
 (45, 1),
 (46, 2),
 (47, 1),
 (48, 1),
 (49, 1),
 (50, 1),
 (51, 1),
 (52, 1),
 (53, 1),
 (54, 1),
 (55, 1),
 (56, 1),
 (57, 2),
 (58, 1),
 (59, 1),
 (60, 1),
 (61, 1),
 (62, 1),
 (63, 2),
 (64, 1),
 (65, 1),
 (66, 1),
 (67, 1),
 (68, 1),
 (69, 1),
 (70, 2),
 (71, 3),
 (72, 1),
 (73, 1),
 (74, 1),
 (75, 2),
 (76, 1),
 (77, 1)]

## From LDA to BERTopic

LDA models documents as **mixtures over topics**, where each topic is a distribution over words (slides 17–20). It's a probabilistic generative model — interpretable, but ignores word order and semantics.

**BERTopic** has the same goal — assign each document a topic — but uses **BERT embeddings + clustering** under the hood. Each document gets a contextual vector representation, then UMAP + HDBSCAN groups similar documents into topics. Result: topics tend to be more semantically coherent without heavy preprocessing.

## BERTopic

In [38]:
%%time

topic_model = BERTopic(min_topic_size=2)
topics, probs = topic_model.fit_transform(df.all_items)

CPU times: user 348 ms, sys: 147 ms, total: 495 ms
Wall time: 1.23 s


In [39]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,2,-1_the_and_to_legal,"[the, and, to, legal, located, dobbs, federal,...","[['The Dobbs Decision: In 2022, the Court rule..."
1,0,46,0_or_the_and_of,"[or, the, and, of, to, in, can, your, for, sym...",[['HIV: A virus that destroys specific white b...
2,1,15,1_feeling_infections_frequent_patches,"[feeling, infections, frequent, patches, skin,...","[[], [], ['Frequent Urination (Polyuria): Urin..."
3,2,8,2_insulin_blood_and_sugar,"[insulin, blood, and, sugar, the, glucose, fre...","[['When you eat, your body breaks down carbohy..."
4,3,7,3_surgery_facial_bottom_and,"[surgery, facial, bottom, and, breast, to, gen...",[['Top Surgery: Includes bilateral mastectomie...
5,4,7,4_the_cupping_and_to,"[the, cupping, and, to, for, suction, is, pain...",[['Dry Cupping: The suction is applied to the ...
6,5,4,5_herbs_and_or_into,"[herbs, and, or, into, to, used, taken, soothe...","[['Ingestion: Brewed as teas, taken in capsule..."
